# 02 — SmarTRIZ DPO Training

Loads preference pairs built by `scripts/build_dpo_pairs.py` and fine-tunes
the SFT model with DPO using the PEFT-no-ref + precompute strategy
(memory-safe on A100 40GB).

**Prerequisites:**
- Notebook 01 complete; `checkpoints/sft-7b/merged/` exists.
- `data/dpo_pairs.jsonl` exists (run the pair-builder cell below or
  `python scripts/build_dpo_pairs.py` outside the notebook).

In [ ]:
# ─── CONFIG — edit this cell only ────────────────────────
import os, sys
from pathlib import Path

MODEL_SIZE = '7b'
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'

# Repo path on Colab — adjust if your repo lives elsewhere
REPO_PATH = '/content/smartriz-project'
if Path(REPO_PATH).exists() and REPO_PATH + '/src' not in sys.path:
    sys.path.insert(0, REPO_PATH + '/src')

# DeepInfra key kept for optional teacher fallback only (not required)
try:
    from google.colab import userdata
    DEEPINFRA_API_KEY = userdata.get('DEEPINFRA_API_KEY') or os.getenv('DEEPINFRA_API_KEY', '')
except Exception:
    DEEPINFRA_API_KEY = os.getenv('DEEPINFRA_API_KEY', '')

SFT_MODEL_DIR    = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/merged/'
OUTPUT_DIR       = f'{DRIVE_PATH}checkpoints/dpo-{MODEL_SIZE}/'
DPO_PAIRS_PATH   = f'{DRIVE_PATH}data/dpo_pairs.jsonl'
JUDGED_PATH      = f'{DRIVE_PATH}data/judged.jsonl'
VALIDATED_PATH   = f'{DRIVE_PATH}data/matrix_validated.jsonl'

# DPO training hyperparameters (per approved plan §3)
LORA_R              = 16
LORA_ALPHA          = 16              # alpha=r is the DPO standard
LORA_DROPOUT        = 0.05
MAX_LENGTH          = 2048
MAX_PROMPT_LENGTH   = 1024
BETA                = 0.1
LEARNING_RATE       = 1e-5            # adapter-tuned DPO; 5e-5 was 10x too high
NUM_EPOCHS          = 2
PER_DEVICE_BSZ      = 1               # 40 GB safety
GRAD_ACCUM_STEPS    = 8               # effective batch = 8
LOSS_TYPE           = 'apo_zero'      # SFT model underperforms winners; fallback: 'sigmoid'
WARMUP_RATIO        = 0.1
LR_SCHEDULER        = 'cosine'

# Pair-builder targets
PAIRS_TOTAL         = 3000
PAIRS_RATIOS        = '0.5,0.3,0.2'   # A:B:C — format/content/matrix

print(f'SFT source:    {SFT_MODEL_DIR}')
print(f'DPO output:    {OUTPUT_DIR}')
print(f'Pairs file:    {DPO_PAIRS_PATH}')
print(f'Loss / LR / β: {LOSS_TYPE} / {LEARNING_RATE} / {BETA}')


In [ ]:
# trl >= 0.11.4 needed for apo_zero + PEFT-no-ref + precompute_ref_log_probs
!pip install -q transformers==4.45.2 peft==0.13.2 trl==0.11.4 \
  bitsandbytes==0.45.0 accelerate==0.34.2 datasets==3.0.1 tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Clone repo so smartriz helpers + scripts are importable ──────────────
# Required because Cell 1 set REPO_PATH but the actual code lives in your
# GitHub repo, not in Drive. Adjust REPO_URL below to match your fork.
#   - Public repo:  https://github.com/<user>/smartriz-project.git
#   - Private repo: https://<TOKEN>@github.com/<user>/smartriz-project.git
import sys
from pathlib import Path

REPO_URL = 'https://github.com/sevketugurel/smartriz-project.git'

if not Path(REPO_PATH).exists():
    print(f'Cloning {REPO_URL} → {REPO_PATH}')
    !git clone --depth 1 {REPO_URL} {REPO_PATH}
else:
    print(f'Repo already present at {REPO_PATH} — pulling latest')
    !cd {REPO_PATH} && git pull --ff-only

if REPO_PATH + '/src' not in sys.path:
    sys.path.insert(0, REPO_PATH + '/src')

# Smoke test — fails loudly if the clone or PYTHONPATH is wrong
from smartriz.training.formatter import SYSTEM_PROMPT  # noqa: F401
from smartriz.eval.format_check import score_format_compliance  # noqa: F401
print('Repo helpers import OK')

In [ ]:
# ── Build DPO pairs (skips if file already exists) ───────────────────────
import json
from pathlib import Path

if Path(DPO_PAIRS_PATH).exists():
    n = sum(1 for _ in open(DPO_PAIRS_PATH))
    print(f'Pairs file already present: {n} pairs at {DPO_PAIRS_PATH}')
else:
    print(f'Building DPO pairs → {DPO_PAIRS_PATH}')
    !python {REPO_PATH}/scripts/build_dpo_pairs.py \
        --judged {JUDGED_PATH} \
        --validated {VALIDATED_PATH} \
        --out {DPO_PAIRS_PATH} \
        --total {PAIRS_TOTAL} \
        --ratios {PAIRS_RATIOS}

# Load
dpo_pairs = [json.loads(l) for l in open(DPO_PAIRS_PATH)]
print(f'Loaded {len(dpo_pairs)} preference pairs')

# Resume detection
checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) \
    if Path(OUTPUT_DIR).exists() else []
RESUME_FROM = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
print(f'DPO checkpoint: {RESUME_FROM or "none — training from scratch"}')


In [ ]:
# ── Load SFT merged model + apply fresh LoRA for DPO (bf16, no bitsandbytes) ──
# bitsandbytes/triton GPU binary unavailable on Colab Py3.12 — use bf16 instead.
# PEFT-no-ref strategy still applies: ref_model=None + peft_config → TRL disables
# the adapter to compute reference logprobs without a second model copy.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules='all-linear',
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Model loaded in bf16. Params: {total:,} total (LoRA will add trainable via peft_config).')


In [ ]:
# ── Build HF dataset and run DPOTrainer ───────────────────────────────────
import sys
from datasets import Dataset as HFDataset
from trl import DPOTrainer, DPOConfig

# Use the shared SmarTRIZ system prompt from the formatter module
sys.path.insert(0, REPO_PATH + '/src')
from smartriz.training.formatter import SYSTEM_PROMPT

def format_dpo_prompt(pair):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': pair['prompt']}],
        tokenize=False, add_generation_prompt=True
    )
    return {'prompt': prompt, 'chosen': pair['chosen'], 'rejected': pair['rejected']}

dpo_hf = HFDataset.from_list([format_dpo_prompt(p) for p in dpo_pairs])
print(f'DPO dataset size: {len(dpo_hf)}')

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=BETA,
    loss_type=LOSS_TYPE,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    per_device_train_batch_size=PER_DEVICE_BSZ,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    bf16=True,
    optim='adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    precompute_ref_log_probs=True,    # frees ref model after one pass
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to='wandb',
    run_name=f'dpo-{MODEL_SIZE}-{LOSS_TYPE}',
)

# ref_model=None + peft_config → adapter-disable used for reference logps;
# this avoids a second 4-bit model copy in VRAM.
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_hf,
    tokenizer=tokenizer,
    peft_config=lora_cfg,
)

dpo_trainer.train(resume_from_checkpoint=RESUME_FROM)
print('DPO training complete')


In [ ]:
# ── Merge DPO LoRA into base and save ─────────────────────────────────────
from pathlib import Path

merged_dpo = OUTPUT_DIR + 'merged/'
if not Path(merged_dpo + 'config.json').exists():
    print('Merging DPO LoRA weights...')
    merged = dpo_trainer.model.merge_and_unload()
    merged.save_pretrained(merged_dpo)
    tokenizer.save_pretrained(merged_dpo)
    print(f'DPO merged model saved to {merged_dpo}')
else:
    print(f'DPO merged model already exists at {merged_dpo}')

print()
print('Ready for Notebook 03 (GGUF conversion + 4-model evaluation)')
